# 🔍 Visualize Initial Frame Variation

This notebook analyzes the variation in initial configurations across episodes in a LeRobot dataset.

**What it does:**
1. Reads the **first frame** from each camera (static, left, right) for every episode
2. **Overlays** all first frames with low alpha to visualize configuration variation
3. **Quantifies** the variation using ResNet18 embeddings (distance matrix + variance)

**Use cases:**
- Check if initial robot/object positions are consistent across episodes
- Identify outlier episodes with unusual starting configurations
- Understand the diversity of your dataset

---
## 1. Configuration

Set the path to your LeRobot dataset.

In [ ]:
import pathlib

# TODO: Set the path to your LeRobot dataset
DATASET_PATH = pathlib.Path("/data/lerobot/your_dataset_here")

# Camera keys to analyze (adjust based on your dataset)
CAMERA_KEYS = [
    "observation.images.rgb_static",
    "observation.images.rgb_left", 
    "observation.images.rgb_right",
]

# Alpha value for overlay visualization (lower = more transparent)
OVERLAY_ALPHA = 0.3

# Maximum episodes to process (None = all)
MAX_EPISODES = None

print(f"Dataset path: {DATASET_PATH}")
print(f"Camera keys: {CAMERA_KEYS}")

---
## 2. Load Dataset and Extract First Frames

In [ ]:
import json
import numpy as np
from PIL import Image
from collections import defaultdict

# Load dataset metadata
info_path = DATASET_PATH / "meta" / "info.json"
episodes_path = DATASET_PATH / "meta" / "episodes.jsonl"

with open(info_path) as f:
    info = json.load(f)

total_episodes = info.get("total_episodes", 0)
fps = info.get("fps", 10)
features = info.get("features", {})

print(f"Dataset info:")
print(f"  Total episodes: {total_episodes}")
print(f"  FPS: {fps}")
print(f"  Available features: {list(features.keys())}")

# Check which camera keys are available
available_cameras = [key for key in CAMERA_KEYS if key in features]
print(f"\nAvailable cameras: {available_cameras}")

if not available_cameras:
    print("\n⚠️  No matching camera keys found! Available image features:")
    for key in features:
        if "image" in key.lower():
            print(f"    - {key}")

In [ ]:
from pathlib import Path
import glob

def load_first_frame_for_episode(dataset_path: Path, episode_idx: int, camera_key: str) -> np.ndarray | None:
    """Load the first frame for a specific episode and camera.
    
    LeRobot v2.0+ stores videos as: videos/chunk-{chunk:03d}/{camera_key}/episode_{ep:06d}.mp4
    Older format: videos/{camera_name}/episode_{ep:06d}.mp4
    Images format: images/{camera_key}/episode_{ep:06d}/frame_{frame:06d}.png
    """
    camera_name = camera_key.split(".")[-1]
    
    # Try different path patterns (ordered by likelihood)
    video_patterns = [
        # LeRobot v2.0+ chunked format (most common)
        dataset_path / "videos" / "chunk-*" / camera_key / f"episode_{episode_idx:06d}.mp4",
        # Full camera key path
        dataset_path / "videos" / camera_key / f"episode_{episode_idx:06d}.mp4",
        # Short camera name path
        dataset_path / "videos" / camera_name / f"episode_{episode_idx:06d}.mp4",
    ]
    
    image_patterns = [
        # Full camera key path
        dataset_path / "images" / camera_key / f"episode_{episode_idx:06d}" / "frame_000000.png",
        # Short camera name path  
        dataset_path / "images" / camera_name / f"episode_{episode_idx:06d}" / "frame_000000.png",
    ]
    
    # Try image paths first (faster to load)
    for img_pattern in image_patterns:
        if img_pattern.exists():
            img = Image.open(img_pattern).convert("RGB")
            return np.array(img)
    
    # Try video paths (use glob for chunk-* pattern)
    for video_pattern in video_patterns:
        # Use glob for patterns with wildcards
        matches = list(glob.glob(str(video_pattern)))
        if matches:
            video_path = matches[0]
            # Use imageio for AV1 decoding (better codec support than cv2)
            try:
                import imageio.v3 as iio
                # Read first frame using ffmpeg backend
                frame = iio.imread(video_path, index=0, plugin="pyav")
                return frame
            except Exception as e1:
                # Fallback to cv2
                try:
                    import cv2
                    cap = cv2.VideoCapture(video_path)
                    ret, frame = cap.read()
                    cap.release()
                    if ret:
                        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                except Exception as e2:
                    pass
            break
    
    return None

# Collect first frames for each camera
first_frames = defaultdict(list)  # camera_key -> list of (episode_idx, image)

num_episodes = min(total_episodes, MAX_EPISODES) if MAX_EPISODES else total_episodes
print(f"Loading first frames for {num_episodes} episodes...")

for ep_idx in range(num_episodes):
    for camera_key in available_cameras:
        frame = load_first_frame_for_episode(DATASET_PATH, ep_idx, camera_key)
        if frame is not None:
            first_frames[camera_key].append((ep_idx, frame))
    
    if (ep_idx + 1) % 10 == 0:
        print(f"  Processed {ep_idx + 1}/{num_episodes} episodes")

print(f"\nLoaded frames:")
for camera_key, frames in first_frames.items():
    print(f"  {camera_key}: {len(frames)} frames")

---
## 3. Visualize Overlaid First Frames

Overlay all first frames with low alpha to see the variation in initial configurations.

In [ ]:
import matplotlib.pyplot as plt

def create_overlay_image(frames: list[tuple[int, np.ndarray]], alpha: float = 0.3) -> np.ndarray:
    """Create an overlay of multiple frames with given alpha.
    
    Args:
        frames: List of (episode_idx, image_array) tuples
        alpha: Transparency for each frame (0-1)
    
    Returns:
        Blended overlay image
    """
    if not frames:
        return None
    
    # Use first frame as reference for shape
    _, ref_frame = frames[0]
    h, w, c = ref_frame.shape
    
    # Accumulator for blending
    overlay = np.zeros((h, w, c), dtype=np.float64)
    
    for _, frame in frames:
        # Resize if needed
        if frame.shape[:2] != (h, w):
            frame = np.array(Image.fromarray(frame).resize((w, h)))
        overlay += frame.astype(np.float64) * alpha
    
    # Normalize to 0-255 range
    overlay = np.clip(overlay / (len(frames) * alpha) * alpha + 
                      overlay / overlay.max() * 255 * (1 - alpha), 0, 255)
    
    # Alternative: simple average
    overlay_avg = np.mean([f[1].astype(np.float64) for f in frames], axis=0)
    
    return overlay_avg.astype(np.uint8)

def create_std_image(frames: list[tuple[int, np.ndarray]]) -> np.ndarray:
    """Create a standard deviation image to highlight areas of high variation."""
    if not frames:
        return None
    
    # Stack all frames
    stacked = np.stack([f[1].astype(np.float64) for f in frames], axis=0)
    
    # Calculate per-pixel standard deviation
    std_img = np.std(stacked, axis=0)
    
    # Normalize for visualization
    std_normalized = (std_img / std_img.max() * 255).astype(np.uint8)
    
    return std_normalized

# Create visualizations for each camera
fig, axes = plt.subplots(len(available_cameras), 3, figsize=(18, 6 * len(available_cameras)))

if len(available_cameras) == 1:
    axes = axes.reshape(1, -1)

for idx, camera_key in enumerate(available_cameras):
    frames = first_frames[camera_key]
    
    if not frames:
        continue
    
    # Sample first frame
    _, sample_frame = frames[0]
    
    # Create overlay (mean)
    overlay = create_overlay_image(frames, OVERLAY_ALPHA)
    
    # Create std deviation image
    std_img = create_std_image(frames)
    
    # Plot
    camera_name = camera_key.split(".")[-1]
    
    axes[idx, 0].imshow(sample_frame)
    axes[idx, 0].set_title(f"{camera_name}\nFirst Episode (Sample)", fontsize=12)
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(overlay)
    axes[idx, 1].set_title(f"{camera_name}\nMean of {len(frames)} First Frames", fontsize=12)
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(std_img)
    axes[idx, 2].set_title(f"{camera_name}\nStd Dev (Bright = High Variation)", fontsize=12)
    axes[idx, 2].axis('off')

plt.suptitle(f"Initial Frame Variation Analysis ({len(frames)} Episodes)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Compute ResNet18 Embeddings

Use a pretrained ResNet18 to compute embeddings for each first frame, then analyze the embedding space.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from tqdm import tqdm

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pretrained ResNet18 and remove the classification head
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet18 = nn.Sequential(*list(resnet18.children())[:-1])  # Remove final FC layer
resnet18 = resnet18.to(device)
resnet18.eval()

# ImageNet normalization
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def compute_embedding(image: np.ndarray) -> np.ndarray:
    """Compute ResNet18 embedding for an image."""
    with torch.no_grad():
        img_tensor = preprocess(image).unsqueeze(0).to(device)
        embedding = resnet18(img_tensor)
        return embedding.squeeze().cpu().numpy()

# Compute embeddings for all first frames
embeddings = defaultdict(list)  # camera_key -> list of (episode_idx, embedding)

print("Computing ResNet18 embeddings...")
for camera_key in available_cameras:
    frames = first_frames[camera_key]
    print(f"\n  {camera_key}: {len(frames)} frames")
    
    for ep_idx, frame in tqdm(frames, desc=f"    {camera_key.split('.')[-1]}"):
        emb = compute_embedding(frame)
        embeddings[camera_key].append((ep_idx, emb))

print("\n✅ Embeddings computed!")

---
## 5. Analyze Embedding Distance and Variance

In [ ]:
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage

def analyze_embeddings(camera_key: str, emb_list: list[tuple[int, np.ndarray]]):
    """Analyze embedding distances and variance for a camera."""
    episode_indices = [e[0] for e in emb_list]
    emb_matrix = np.stack([e[1] for e in emb_list])  # (N, 512)
    
    # Compute pairwise distances
    distances = pdist(emb_matrix, metric='cosine')
    distance_matrix = squareform(distances)
    
    # Compute statistics
    mean_distance = np.mean(distances)
    std_distance = np.std(distances)
    max_distance = np.max(distances)
    min_distance = np.min(distances[distances > 0]) if np.any(distances > 0) else 0
    
    # Embedding variance (per dimension, then aggregate)
    emb_variance = np.var(emb_matrix, axis=0)
    total_variance = np.sum(emb_variance)
    mean_variance = np.mean(emb_variance)
    
    # Average distance to all other embeddings
    avg_distances = np.mean(distance_matrix, axis=1)
    
    # Distance to closest embedding (excluding self = 0)
    # Set diagonal to inf so we don't pick self as closest
    dist_matrix_no_self = distance_matrix.copy()
    np.fill_diagonal(dist_matrix_no_self, np.inf)
    min_distances = np.min(dist_matrix_no_self, axis=1)
    
    # Find outliers using BOTH criteria:
    # 1. High average distance (original method)
    avg_outlier_threshold = mean_distance + 2 * std_distance
    avg_outlier_indices = set(np.where(avg_distances > avg_outlier_threshold)[0])
    
    # 2. High minimum distance (new method - no close neighbors)
    mean_min_dist = np.mean(min_distances)
    std_min_dist = np.std(min_distances)
    min_outlier_threshold = mean_min_dist + 2 * std_min_dist
    min_outlier_indices = set(np.where(min_distances > min_outlier_threshold)[0])
    
    # Combine outliers from both methods
    all_outlier_indices = avg_outlier_indices | min_outlier_indices
    outlier_episodes = [episode_indices[i] for i in all_outlier_indices]
    
    # Track which method flagged each outlier
    outlier_reasons = {}
    for i in all_outlier_indices:
        ep = episode_indices[i]
        reasons = []
        if i in avg_outlier_indices:
            reasons.append("high_avg_dist")
        if i in min_outlier_indices:
            reasons.append("high_min_dist")
        outlier_reasons[ep] = reasons
    
    return {
        'distance_matrix': distance_matrix,
        'episode_indices': episode_indices,
        'mean_distance': mean_distance,
        'std_distance': std_distance,
        'max_distance': max_distance,
        'min_distance': min_distance,
        'total_variance': total_variance,
        'mean_variance': mean_variance,
        'avg_distances': avg_distances,
        'min_distances': min_distances,
        'mean_min_distance': mean_min_dist,
        'std_min_distance': std_min_dist,
        'outlier_episodes': outlier_episodes,
        'outlier_reasons': outlier_reasons,
        'emb_matrix': emb_matrix,
    }

# Analyze each camera
analysis_results = {}

print("="*70)
print("EMBEDDING ANALYSIS RESULTS")
print("="*70)

for camera_key in available_cameras:
    emb_list = embeddings[camera_key]
    if len(emb_list) < 2:
        print(f"\n⚠️  {camera_key}: Not enough embeddings for analysis")
        continue
    
    results = analyze_embeddings(camera_key, emb_list)
    analysis_results[camera_key] = results
    
    camera_name = camera_key.split('.')[-1]
    print(f"\n📷 {camera_name}")
    print(f"   ├─ Episodes analyzed: {len(emb_list)}")
    print(f"   ├─ Cosine Distance (avg to others):")
    print(f"   │    Mean: {results['mean_distance']:.4f}")
    print(f"   │    Std:  {results['std_distance']:.4f}")
    print(f"   │    Range: [{results['min_distance']:.4f}, {results['max_distance']:.4f}]")
    print(f"   ├─ Cosine Distance (to closest):")
    print(f"   │    Mean: {results['mean_min_distance']:.4f}")
    print(f"   │    Std:  {results['std_min_distance']:.4f}")
    print(f"   ├─ Embedding Variance:")
    print(f"   │    Total: {results['total_variance']:.4f}")
    print(f"   │    Mean per dim: {results['mean_variance']:.6f}")
    if results['outlier_episodes']:
        print(f"   └─ ⚠️  Potential outliers:")
        for ep in results['outlier_episodes']:
            reasons = results['outlier_reasons'][ep]
            reason_str = ", ".join(reasons)
            print(f"         Episode {ep}: [{reason_str}]")
    else:
        print(f"   └─ ✅ No significant outliers detected")

print("\n" + "="*70)

---
## 6. Visualize Distance Matrix and Clustering

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for camera_key, results in analysis_results.items():
    camera_name = camera_key.split('.')[-1]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Distance Matrix Heatmap
    im = axes[0, 0].imshow(results['distance_matrix'], cmap='viridis', aspect='auto')
    axes[0, 0].set_title(f'Pairwise Cosine Distance Matrix', fontsize=12)
    axes[0, 0].set_xlabel('Episode Index')
    axes[0, 0].set_ylabel('Episode Index')
    plt.colorbar(im, ax=axes[0, 0], label='Cosine Distance')
    
    # 2. Average Distance per Episode (bar plot)
    episode_indices = results['episode_indices']
    avg_distances = results['avg_distances']
    
    colors_avg = ['red' if ep in results['outlier_episodes'] else 'steelblue' 
                  for ep in episode_indices]
    
    axes[0, 1].bar(range(len(episode_indices)), avg_distances, color=colors_avg, alpha=0.7)
    axes[0, 1].axhline(y=results['mean_distance'], color='green', linestyle='--', 
                       label=f"Mean: {results['mean_distance']:.3f}")
    axes[0, 1].axhline(y=results['mean_distance'] + 2*results['std_distance'], 
                       color='orange', linestyle=':', label='Threshold (2σ)')
    axes[0, 1].set_title(f'Average Distance to Other Episodes', fontsize=12)
    axes[0, 1].set_xlabel('Episode Index')
    axes[0, 1].set_ylabel('Avg Cosine Distance')
    axes[0, 1].legend(fontsize=8)
    
    # 3. Minimum Distance per Episode (bar plot) - NEW
    min_distances = results['min_distances']
    
    colors_min = ['red' if ep in results['outlier_episodes'] else 'steelblue' 
                  for ep in episode_indices]
    
    axes[1, 0].bar(range(len(episode_indices)), min_distances, color=colors_min, alpha=0.7)
    axes[1, 0].axhline(y=results['mean_min_distance'], color='green', linestyle='--', 
                       label=f"Mean: {results['mean_min_distance']:.3f}")
    axes[1, 0].axhline(y=results['mean_min_distance'] + 2*results['std_min_distance'], 
                       color='orange', linestyle=':', label='Threshold (2σ)')
    axes[1, 0].set_title(f'Distance to Closest Episode (Nearest Neighbor)', fontsize=12)
    axes[1, 0].set_xlabel('Episode Index')
    axes[1, 0].set_ylabel('Min Cosine Distance')
    axes[1, 0].legend(fontsize=8)
    
    # 4. Hierarchical Clustering Dendrogram
    linkage_matrix = linkage(results['emb_matrix'], method='ward')
    dendrogram(linkage_matrix, ax=axes[1, 1], labels=episode_indices, leaf_rotation=90)
    axes[1, 1].set_title(f'Hierarchical Clustering', fontsize=12)
    axes[1, 1].set_xlabel('Episode Index')
    axes[1, 1].set_ylabel('Distance')
    
    plt.suptitle(f'Embedding Analysis: {camera_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 7. t-SNE Visualization of Embeddings

Reduce embeddings to 2D for visualization using t-SNE.

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(1, len(analysis_results), figsize=(6 * len(analysis_results), 5))

if len(analysis_results) == 1:
    axes = [axes]

for idx, (camera_key, results) in enumerate(analysis_results.items()):
    camera_name = camera_key.split('.')[-1]
    emb_matrix = results['emb_matrix']
    episode_indices = results['episode_indices']
    
    # Apply t-SNE
    perplexity = min(30, len(emb_matrix) - 1)  # Adjust perplexity for small datasets
    tsne = TSNE(n_components=2, random_state=42, perplexity=max(5, perplexity))
    emb_2d = tsne.fit_transform(emb_matrix)
    
    # Plot
    scatter = axes[idx].scatter(emb_2d[:, 0], emb_2d[:, 1], 
                                 c=episode_indices, cmap='viridis', 
                                 s=100, alpha=0.7, edgecolors='white')
    
    # Highlight outliers
    outlier_mask = [ep in results['outlier_episodes'] for ep in episode_indices]
    if any(outlier_mask):
        outlier_points = emb_2d[outlier_mask]
        axes[idx].scatter(outlier_points[:, 0], outlier_points[:, 1], 
                          c='red', s=200, marker='x', linewidths=3, label='Outliers')
        axes[idx].legend()
    
    # Add episode labels
    for i, ep_idx in enumerate(episode_indices):
        axes[idx].annotate(str(ep_idx), (emb_2d[i, 0], emb_2d[i, 1]), 
                           fontsize=8, ha='center', va='bottom')
    
    axes[idx].set_title(f'{camera_name}\nt-SNE Embedding Visualization', fontsize=12)
    axes[idx].set_xlabel('t-SNE Dimension 1')
    axes[idx].set_ylabel('t-SNE Dimension 2')
    plt.colorbar(scatter, ax=axes[idx], label='Episode Index')

plt.suptitle('t-SNE Visualization of First Frame Embeddings', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Show Individual Outlier Frames

Display the first frames from episodes identified as outliers.

In [ ]:
# Collect all outliers across cameras
all_outliers = set()
for camera_key, results in analysis_results.items():
    all_outliers.update(results['outlier_episodes'])

if all_outliers:
    from matplotlib.backends.backend_pdf import PdfPages
    
    # Output path for PDF
    PDF_OUTPUT_PATH = pathlib.Path("outputs") / f"outlier_analysis_{DATASET_PATH.name}.pdf"
    PDF_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"Showing first frames from outlier episodes: {sorted(all_outliers)}")
    print(f"Comparing against: 1) Mean of all frames, 2) Closest neighbor")
    print(f"Saving to: {PDF_OUTPUT_PATH}\n")
    
    # Compute mean frame for each camera
    mean_frames = {}
    for camera_key in available_cameras:
        frames = first_frames[camera_key]
        if frames:
            stacked = np.stack([f[1].astype(np.float64) for f in frames], axis=0)
            mean_frames[camera_key] = np.mean(stacked, axis=0).astype(np.uint8)
    
    # Find closest neighbor for each outlier using the distance matrix
    def find_closest_neighbor(outlier_ep: int, camera_key: str) -> int | None:
        """Find the episode index of the closest neighbor to an outlier."""
        if camera_key not in analysis_results:
            return None
        
        results = analysis_results[camera_key]
        episode_indices = results['episode_indices']
        distance_matrix = results['distance_matrix']
        
        if outlier_ep not in episode_indices:
            return None
        
        outlier_idx = episode_indices.index(outlier_ep)
        
        # Get distances to all other episodes (set self to inf)
        distances = distance_matrix[outlier_idx].copy()
        distances[outlier_idx] = np.inf
        
        # Find closest
        closest_idx = np.argmin(distances)
        return episode_indices[closest_idx]
    
    # Build frame lookup for quick access
    frame_lookup = {}
    for camera_key in available_cameras:
        frame_lookup[camera_key] = {ep_idx: frame for ep_idx, frame in first_frames[camera_key]}
    
    with PdfPages(PDF_OUTPUT_PATH) as pdf:
        # Create TWO pages per outlier episode: one vs mean, one vs closest
        for outlier_ep in sorted(all_outliers):
            n_cameras = len(available_cameras)
            
            # Get outlier reasons for titles
            reasons_all = []
            for camera_key, results in analysis_results.items():
                if outlier_ep in results.get('outlier_reasons', {}):
                    reasons_all.extend(results['outlier_reasons'][outlier_ep])
            reasons_str = ", ".join(set(reasons_all)) if reasons_all else "detected"
            
            # ========== PAGE 1: Compare to MEAN ==========
            fig1, axes1 = plt.subplots(n_cameras, 4, figsize=(16, 4 * n_cameras))
            if n_cameras == 1:
                axes1 = axes1.reshape(1, -1)
            
            for cam_idx, camera_key in enumerate(available_cameras):
                camera_name = camera_key.split('.')[-1]
                outlier_frame = frame_lookup[camera_key].get(outlier_ep)
                mean_frame = mean_frames.get(camera_key)
                
                if outlier_frame is not None and mean_frame is not None:
                    # 1. Outlier frame
                    axes1[cam_idx, 0].imshow(outlier_frame)
                    axes1[cam_idx, 0].set_title(f'{camera_name}\nOutlier (Ep {outlier_ep})', 
                                                fontsize=14, fontweight='bold')
                    axes1[cam_idx, 0].axis('off')
                    
                    # 2. Overlay of outlier and mean (blended)
                    blend_alpha = 0.5
                    blended = (outlier_frame.astype(np.float64) * blend_alpha + 
                              mean_frame.astype(np.float64) * (1 - blend_alpha)).astype(np.uint8)
                    axes1[cam_idx, 1].imshow(blended)
                    axes1[cam_idx, 1].set_title(f'{camera_name}\nOverlay (Outlier + Mean)', fontsize=14)
                    axes1[cam_idx, 1].axis('off')
                    
                    # 3. Difference heatmap
                    diff = np.abs(outlier_frame.astype(np.float64) - mean_frame.astype(np.float64))
                    diff_gray = np.mean(diff, axis=2)
                    diff_normalized = diff_gray / diff_gray.max() if diff_gray.max() > 0 else diff_gray
                    
                    axes1[cam_idx, 2].imshow(diff_normalized, cmap='hot', vmin=0, vmax=1)
                    axes1[cam_idx, 2].set_title(f'{camera_name}\nDifference Heatmap', fontsize=14)
                    axes1[cam_idx, 2].axis('off')
                    
                    # 4. Outlier with overlay
                    threshold = np.percentile(diff_gray, 90)
                    mask = diff_gray > threshold
                    overlay = outlier_frame.copy().astype(np.float64)
                    alpha = 0.5
                    overlay[mask, 0] = overlay[mask, 0] * (1 - alpha) + 255 * alpha
                    overlay[mask, 1] = overlay[mask, 1] * (1 - alpha) + 0 * alpha
                    overlay[mask, 2] = overlay[mask, 2] * (1 - alpha) + 255 * alpha
                    
                    axes1[cam_idx, 3].imshow(overlay.astype(np.uint8))
                    axes1[cam_idx, 3].set_title(f'{camera_name}\nDiff Overlay (magenta)', fontsize=14)
                    axes1[cam_idx, 3].axis('off')
                else:
                    for col in range(4):
                        axes1[cam_idx, col].text(0.5, 0.5, 'No frame', ha='center', va='center', fontsize=14)
                        axes1[cam_idx, col].axis('off')
            
            fig1.suptitle(f'Outlier Episode {outlier_ep} vs MEAN\nReason: {reasons_str}', 
                         fontsize=18, fontweight='bold', y=1.02)
            plt.tight_layout()
            pdf.savefig(fig1, bbox_inches='tight')
            plt.show()
            plt.close(fig1)
            
            # ========== PAGE 2: Compare to CLOSEST NEIGHBOR ==========
            fig2, axes2 = plt.subplots(n_cameras, 4, figsize=(16, 4 * n_cameras))
            if n_cameras == 1:
                axes2 = axes2.reshape(1, -1)
            
            for cam_idx, camera_key in enumerate(available_cameras):
                camera_name = camera_key.split('.')[-1]
                outlier_frame = frame_lookup[camera_key].get(outlier_ep)
                
                # Find and get closest neighbor frame
                closest_ep = find_closest_neighbor(outlier_ep, camera_key)
                closest_frame = frame_lookup[camera_key].get(closest_ep) if closest_ep is not None else None
                
                # Get distance to closest neighbor
                closest_dist = None
                if camera_key in analysis_results and closest_ep is not None:
                    results = analysis_results[camera_key]
                    ep_indices = results['episode_indices']
                    if outlier_ep in ep_indices:
                        closest_dist = results['min_distances'][ep_indices.index(outlier_ep)]
                
                if outlier_frame is not None and closest_frame is not None:
                    # 1. Outlier frame
                    axes2[cam_idx, 0].imshow(outlier_frame)
                    axes2[cam_idx, 0].set_title(f'{camera_name}\nOutlier (Ep {outlier_ep})', 
                                                fontsize=14, fontweight='bold')
                    axes2[cam_idx, 0].axis('off')
                    
                    # 2. Overlay of outlier and closest (blended)
                    dist_str = f" (d={closest_dist:.3f})" if closest_dist is not None else ""
                    blend_alpha = 0.5
                    blended = (outlier_frame.astype(np.float64) * blend_alpha + 
                              closest_frame.astype(np.float64) * (1 - blend_alpha)).astype(np.uint8)
                    axes2[cam_idx, 1].imshow(blended)
                    axes2[cam_idx, 1].set_title(f'{camera_name}\nOverlay (Outlier + Ep{closest_ep}){dist_str}', 
                                                fontsize=14)
                    axes2[cam_idx, 1].axis('off')
                    
                    # 3. Difference heatmap
                    diff = np.abs(outlier_frame.astype(np.float64) - closest_frame.astype(np.float64))
                    diff_gray = np.mean(diff, axis=2)
                    diff_normalized = diff_gray / diff_gray.max() if diff_gray.max() > 0 else diff_gray
                    
                    axes2[cam_idx, 2].imshow(diff_normalized, cmap='hot', vmin=0, vmax=1)
                    axes2[cam_idx, 2].set_title(f'{camera_name}\nDifference Heatmap', fontsize=14)
                    axes2[cam_idx, 2].axis('off')
                    
                    # 4. Outlier with overlay
                    threshold = np.percentile(diff_gray, 90)
                    mask = diff_gray > threshold
                    overlay = outlier_frame.copy().astype(np.float64)
                    alpha = 0.5
                    overlay[mask, 0] = overlay[mask, 0] * (1 - alpha) + 255 * alpha
                    overlay[mask, 1] = overlay[mask, 1] * (1 - alpha) + 0 * alpha
                    overlay[mask, 2] = overlay[mask, 2] * (1 - alpha) + 255 * alpha
                    
                    axes2[cam_idx, 3].imshow(overlay.astype(np.uint8))
                    axes2[cam_idx, 3].set_title(f'{camera_name}\nDiff Overlay (magenta)', fontsize=14)
                    axes2[cam_idx, 3].axis('off')
                else:
                    for col in range(4):
                        axes2[cam_idx, col].text(0.5, 0.5, 'No frame', ha='center', va='center', fontsize=14)
                        axes2[cam_idx, col].axis('off')
            
            fig2.suptitle(f'Outlier Episode {outlier_ep} vs CLOSEST NEIGHBOR\nReason: {reasons_str}', 
                         fontsize=18, fontweight='bold', y=1.02)
            plt.tight_layout()
            pdf.savefig(fig2, bbox_inches='tight')
            plt.show()
            plt.close(fig2)
    
    print(f"\n📄 Outlier analysis saved to: {PDF_OUTPUT_PATH}")
    
    # Print summary of difference magnitudes
    print("\n📊 Outlier Difference Summary:")
    print("-" * 80)
    for outlier_ep in sorted(all_outliers):
        print(f"\nEpisode {outlier_ep}:")
        for camera_key in available_cameras:
            camera_name = camera_key.split('.')[-1]
            outlier_frame = frame_lookup[camera_key].get(outlier_ep)
            mean_frame = mean_frames.get(camera_key)
            closest_ep = find_closest_neighbor(outlier_ep, camera_key)
            closest_frame = frame_lookup[camera_key].get(closest_ep) if closest_ep else None
            
            if outlier_frame is not None:
                # vs Mean
                if mean_frame is not None:
                    diff_mean = np.mean(np.abs(outlier_frame.astype(np.float64) - mean_frame.astype(np.float64)))
                else:
                    diff_mean = None
                
                # vs Closest
                if closest_frame is not None:
                    diff_closest = np.mean(np.abs(outlier_frame.astype(np.float64) - closest_frame.astype(np.float64)))
                    emb_dist = None
                    if camera_key in analysis_results:
                        results = analysis_results[camera_key]
                        ep_indices = results['episode_indices']
                        if outlier_ep in ep_indices:
                            emb_dist = results['min_distances'][ep_indices.index(outlier_ep)]
                else:
                    diff_closest = None
                    emb_dist = None
                
                mean_str = f"vs_mean={diff_mean:.1f}" if diff_mean else "N/A"
                closest_str = f"vs_closest(Ep{closest_ep})={diff_closest:.1f}" if diff_closest else "N/A"
                emb_str = f", emb_d={emb_dist:.4f}" if emb_dist else ""
                print(f"  {camera_name}: {mean_str}, {closest_str}{emb_str}")
else:
    print("✅ No outliers detected - all episodes have similar initial configurations!")

---
## 9. Summary Statistics

In [ ]:
import pandas as pd

# Create summary dataframe
summary_data = []

for camera_key, results in analysis_results.items():
    camera_name = camera_key.split('.')[-1]
    summary_data.append({
        'Camera': camera_name,
        'Episodes': len(results['episode_indices']),
        'Mean Distance': f"{results['mean_distance']:.4f}",
        'Std Distance': f"{results['std_distance']:.4f}",
        'Max Distance': f"{results['max_distance']:.4f}",
        'Total Variance': f"{results['total_variance']:.4f}",
        'Outliers': len(results['outlier_episodes']),
        'Outlier Episodes': str(results['outlier_episodes']) if results['outlier_episodes'] else 'None',
    })

df_summary = pd.DataFrame(summary_data)

print("\n" + "="*80)
print("📊 SUMMARY: Initial Frame Variation Analysis")
print("="*80)
print(f"\nDataset: {DATASET_PATH.name}")
print(f"Episodes analyzed: {num_episodes}")
print(f"\n{df_summary.to_string(index=False)}")
print("\n" + "="*80)

# Interpretation
print("\n📝 Interpretation Guide:")
print("   • Low mean distance (< 0.1): Very consistent initial configurations")
print("   • Medium mean distance (0.1 - 0.3): Moderate variation, typical for real-world data")
print("   • High mean distance (> 0.3): High variation, may indicate diverse starting poses")
print("   • Outliers: Episodes with unusually different initial configurations")

---
## ✅ Done!

This analysis helps you understand:
- **Visual consistency**: Are objects/robot in similar positions at the start of each episode?
- **Outlier detection**: Which episodes have unusual starting configurations?
- **Dataset diversity**: How varied are your initial conditions?

**Next steps:**
- If outliers are found, review those episodes for potential data quality issues
- High variance might indicate need for more training data or data augmentation
- Low variance confirms consistent data collection setup